# Post-training epoch 100: geometry
Private exploratory evaluation; fixed settings, no training.


In [ ]:
import os, sys, json, hashlib
from pathlib import Path
for key in ['OMP_NUM_THREADS','MKL_NUM_THREADS','OPENBLAS_NUM_THREADS','NUMEXPR_NUM_THREADS']:
    os.environ[key]='2'
root=Path('/kaggle/input')
hits=list(root.rglob('models_manifest.json'))
assert len(hits)==1, hits
code=hits[0].parent
manifest=json.loads(hits[0].read_text())
for name,expected in manifest['code_hashes'].items():
    assert hashlib.sha256((code/name).read_bytes()).hexdigest()==expected, name
sys.path.insert(0,str(code))
from artifacts_io import unpack
# Real directories + read-only file symlinks preserve benchmark paths; rglob does
# not follow directory symlinks consistently across Python versions.
view=Path('/kaggle/working/input_view');view.mkdir(exist_ok=True)
for source in root.rglob('*'):
    if source.is_file():
        dest=view/'mounted'/source.relative_to(root);dest.parent.mkdir(parents=True,exist_ok=True)
        if not dest.exists():dest.symlink_to(source)
for index,artifact in enumerate(sorted(root.rglob('artifact_manifest.json'))):
    unpack(artifact,view/'dependencies'/str(index))
import runtime

CONFIG = {'stage': 'geometry', 'job_id': 'iir-post100-geometry-paris6k-2', 'input': '/kaggle/working/input_view', 'output': '/kaggle/working', 'dataset': 'paris6k', 'shard_index': 2, 'shard_count': 4}
try:
    runtime.run(CONFIG)
finally:
    import shutil
    shutil.rmtree(view, ignore_errors=True)
